# MoBot: visual navigation and docking with an EKF and LQR (Track A)

A TurtleBot3 Burger in a MuJoCo room (three boxes, a sofa and a coffee table) has to find the charging dock marked with an ArUco tag and park in front of it.

**Pipeline:** camera + LiDAR + wheel encoders + gyro → **EKF** (odometry prediction, ArUco pose update) → planner (search / explore / approach / align / dock) → **LQR** line-tracking controller → wheels.

**Ground truth rule:** the robot code in `src/estimator.py`, `src/planner.py`, `src/control.py`, `src/marker.py`, `src/model.py` only sees sensor readings. The simulator's true pose (`src/judge.py`, `eval/`) is used **only** to score runs and draw plots.

Run the cells in order (**Runtime → Run all** works). Works on the free CPU runtime; a GPU runtime renders faster.

In [ ]:
import os, shutil
if not os.path.exists("Mobile-robot"):
    !git clone -q https://github.com/anne-sharon/Mobile-robot.git
%cd Mobile-robot
!pip install -q mujoco opencv-python matplotlib scipy imageio-ffmpeg

# Pick an offscreen renderer: EGL on a GPU runtime, software OSMesa on a CPU runtime.
if shutil.which("nvidia-smi"):
    icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
    if not os.path.exists(icd):
        os.makedirs(os.path.dirname(icd), exist_ok=True)
        with open(icd, "w") as f:
            f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
    os.environ["MUJOCO_GL"] = "egl"
else:
    !apt-get -qq update
    !apt-get -qq install -y --fix-missing libosmesa6-dev > /dev/null
    os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = os.environ["MUJOCO_GL"]
print("renderer:", os.environ["MUJOCO_GL"])

## A. Model: controllability and observability
Unicycle model of the TurtleBot3, the linearised path-tracking error model used for the LQR, and the camera measurement model used by the EKF (all in `src/model.py`).

In [ ]:
from src.model import analysis, lqr, tracking_error_model
from src.control import Q_LQR, R_LQR
for k, v in analysis().items():
    print(f"{k:45s} rank {v}")
print()
for v in (0.05, 0.12, 0.20):
    K, poles = lqr(*tracking_error_model(v), Q_LQR, R_LQR)
    print(f"LQR at v = {v:.2f} m/s:  K = {K.round(2).ravel()}   closed-loop poles = {poles.real.round(2)}")

## B. Sensing: camera calibration and measurement noise
1. Checkerboard calibration (Zhang's method) of the robot's camera → `models/camera_calibration.json` (used by the robot).
2. ArUco pose error against distance, from 400 known poses → camera bias correction and the EKF's measurement noise R (`models/camera_noise.json`). About 30 s.

In [ ]:
import json
from eval.calibrate_camera import calibrate
from eval.characterise_camera import characterise
cal = calibrate()
print(f"calibration: {cal['views_used']} views, RMS reprojection error {cal['rms_reprojection_px']:.3f} px, "
      f"focal length error {cal['focal_error_pct']:.3f} %")
noise = characterise()
print(json.dumps({k: noise[k] for k in ('bias_c', 'ahead', 'left', 'angle', 'angle_outlier_rate')}, indent=1))

## C + D. One docking run: path, EKF error and uncertainty
Left: robot path coloured by planner stage. Below: EKF error (estimate − truth) with its ±2σ band, and NEES (consistency).

In [ ]:
from eval.plot_episode import run_and_plot
from eval.plot_ekf import plot_error
print(run_and_plot())
print(plot_error())

## Video: nominal run
Overhead view (coloured = true path, white = EKF estimate with its 2σ ellipse), the robot's camera, and the estimation error with its ±2σ band. About 2 minutes to render on CPU. To keep a video, open the **Files** panel → `Mobile-robot/eval/results` → right-click → **Download**.

In [ ]:
from IPython.display import Video, display
from eval.make_video import make_video
out, result = make_video("nominal")
print(result)
display(Video(out, embed=True, width=960))

## Video: disturbed run
When the robot first comes within 1.5 m of the dock it is shoved sideways (the wheels slip, so odometry doesn't see it). Watch the error jump, the uncertainty grow, and the camera pull the estimate back. Try `"occlusion"` (camera blocked for 8 s) or `"combined"` too.

In [ ]:
out, result = make_video("push")
print(result)
display(Video(out, embed=True, width=960))

## E. Robustness study (optional, slow)
Repeated trials from 20 start poses under each condition: nominal, gyro noise + bias, camera pixel noise, push, occlusion, combined. The full study is 120 runs (about 20–30 min on Colab); this cell runs a quick version with 5 starts and 3 conditions.

In [ ]:
from eval import run_batch
summary = run_batch.main(["nominal", "push", "occlusion"], n=5)

## Official score
`eval/run_episode.py`: the judge's ground truth is only used for scoring.

In [ ]:
from eval.run_episode import run_episode
print(run_episode())